In [7]:
# This notebook evaluates models using 10 repeated splits with wealth-inclusive features
def enforce_model_dtypes(df):
    df = df.copy()

    # -------- Category columns --------
    category_cols = [
        "hv270WealthIndex",
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
        "v717Occupation",
        "745aHouseOwnership",
    ]

    for col in category_cols:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # -------- Integer columns --------
    int8_cols = [
        "hv220AgeOfHead",
        "hv009FamilySize",
        "hv014NoOfChildren",
        "EnergyPoor",
    ]

    for col in int8_cols:
        if col in df.columns:
            df[col] = df[col].astype("int8")

    # -------- Float columns --------
    float_cols = [
        "HouseholdClusterElevation",
        "hv216HouseSize",
    ]

    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype("float64")

    return df


In [8]:
import pandas as pd
from autogluon.tabular import TabularPredictor

SPLITS_ROOT = r"MYANMAR/DATA/PAIRTEST10"

def evaluate_saved_models(country, trad_key, dl_key="NN_TORCH", n_splits=10):
    rows = []

    for i in range(1, n_splits + 1):
        # test data
        test_path = f"{SPLITS_ROOT}/split{i}_test.parquet"
        test_df = pd.read_parquet(test_path)
         # ---- Enforce dtypes----
        test_df  = enforce_model_dtypes(test_df)

        # saved model folders (based on your structure)
        trad_path = rf"{SPLITS_ROOT}\MODELS\WEALTH_INCLUSIVE\split{i}\TRAD_{trad_key}"
        dl_path   = rf"{SPLITS_ROOT}\MODELS\WEALTH_INCLUSIVE\split{i}\DL_{dl_key}"

        # load predictors
        pred_tr = TabularPredictor.load(trad_path)
        pred_dl = TabularPredictor.load(dl_path)

        # evaluate
        m_tr = pred_tr.evaluate(test_df, silent=True)
        m_dl = pred_dl.evaluate(test_df, silent=True)
        
        acc_tr = float(m_tr["accuracy"])
        acc_dl = float(m_dl["accuracy"])
        
        mcc_tr = float(m_tr["mcc"])
        mcc_dl = float(m_dl["mcc"])
        
        balacc_tr = float(m_tr["balanced_accuracy"])
        balacc_dl = float(m_dl["balanced_accuracy"])
        
        f1_tr = float(m_tr["f1"])
        f1_dl = float(m_dl["f1"])

        rows.append({
            "Country": country,
            "Split": i,
            "Traditional_Model": trad_key,
            "DL_Model": dl_key,
        
            "Traditional_Accuracy": acc_tr,
            "DL_Accuracy": acc_dl,
            "Diff_Accuracy": acc_tr - acc_dl,
        
            "Traditional_MCC": mcc_tr,
            "DL_MCC": mcc_dl,
            "Diff_MCC": mcc_tr - mcc_dl,
        
            "Traditional_BalancedAcc": balacc_tr,
            "DL_BalancedAcc": balacc_dl,
        
            "Traditional_F1": f1_tr,
            "DL_F1": f1_dl,
        })

        print(f"{country} split {i}: TRAD={acc_tr:.4f}, DL={acc_dl:.4f}")

    return pd.DataFrame(rows)


In [9]:
import os
import numpy as np
from scipy.stats import ttest_rel

RESULTS_FOLDER = rf"{SPLITS_ROOT}\RESULTS"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

country_trad = {
    "MYANMAR": "CAT",
}

all_df = []
print("\n================ SPLIT-LEVEL RESULTS ================\n")
for country, trad_key in country_trad.items():
    df_country = evaluate_saved_models(country, trad_key, dl_key="NN_TORCH", n_splits=10)
    all_df.append(df_country)

df_splits = pd.concat(all_df, ignore_index=True)

#print("\n================ SPLIT-LEVEL RESULTS ================\n")
#print(df_splits)

def mean_std(x):
    return f"{x.mean():.3f} ± {x.std():.3f}"

table6_rows = []

for country in df_splits["Country"].unique():
    sub = df_splits[df_splits["Country"] == country]

    ml_model = sub["Traditional_Model"].iloc[0]
    dl_model = sub["DL_Model"].iloc[0]

    table6_rows.append({
        "Country": country,
        "Model": ml_model,

        "Accuracy (mean ± SD)": mean_std(sub["Traditional_Accuracy"]),
        "MCC (mean ± SD)": mean_std(sub["Traditional_MCC"]),
        "Balanced Accuracy (mean ± SD)": mean_std(sub["Traditional_BalancedAcc"]),
        "F1-Score (mean ± SD)": mean_std(sub["Traditional_F1"]),
    })

    table6_rows.append({
        "Country": country,
        "Model": dl_model,

        "Accuracy (mean ± SD)": mean_std(sub["DL_Accuracy"]),
        "MCC (mean ± SD)": mean_std(sub["DL_MCC"]),
        "Balanced Accuracy (mean ± SD)": mean_std(sub["DL_BalancedAcc"]),
        "F1-Score (mean ± SD)": mean_std(sub["DL_F1"]),
    })
print("\n================ AVERAGE RESULTS ================\n")
df_table6 = pd.DataFrame(table6_rows)
df_table6


================ SPLIT-LEVEL RESULTS ================

MYANMAR split 1: TRAD=0.8981, DL=0.8961
MYANMAR split 2: TRAD=0.8921, DL=0.8897
MYANMAR split 3: TRAD=0.8889, DL=0.8881
MYANMAR split 4: TRAD=0.8953, DL=0.8953
MYANMAR split 5: TRAD=0.8885, DL=0.8889
MYANMAR split 6: TRAD=0.9034, DL=0.8973
MYANMAR split 7: TRAD=0.8961, DL=0.9010
MYANMAR split 8: TRAD=0.9066, DL=0.8977
MYANMAR split 9: TRAD=0.8945, DL=0.8865
MYANMAR split 10: TRAD=0.8957, DL=0.8881

================ AVERAGE RESULTS ================



,Country,Model,Accuracy (mean ± SD),MCC (mean ± SD),Balanced Accuracy (mean ± SD),F1-Score (mean ± SD)
0,MYANMAR,CAT,0.896 ± 0.006,0.690 ± 0.017,0.834 ± 0.010,0.934 ± 0.004
1,MYANMAR,NN_TORCH,0.893 ± 0.005,0.685 ± 0.016,0.837 ± 0.011,0.932 ± 0.003


In [10]:
save_confirm = True   # change to False if don't want to save yet
if save_confirm:
    country_folder = rf"{RESULTS_FOLDER}\{country}"
    os.makedirs(country_folder, exist_ok=True)
    
    df_table6.to_csv(rf"{country_folder}\Table6_myanmar_best_performing_wealth_inclusive.csv", index=False)
    print("\nFiles saved successfully.")


Files saved successfully.


In [11]:
from scipy.stats import ttest_rel
import numpy as np

ttest_rows = []

for country in df_splits["Country"].unique():

    sub = df_splits[df_splits["Country"] == country].sort_values("Split")

    ml_model = sub["Traditional_Model"].iloc[0]
    dl_model = sub["DL_Model"].iloc[0]

    # ---------------- ACCURACY ----------------
    trad_acc = sub["Traditional_Accuracy"].to_numpy()
    dl_acc   = sub["DL_Accuracy"].to_numpy()

    diff_acc = trad_acc - dl_acc
    t_acc, p_acc = ttest_rel(trad_acc, dl_acc)

    # ---------------- MCC ----------------
    trad_mcc = sub["Traditional_MCC"].to_numpy()
    dl_mcc   = sub["DL_MCC"].to_numpy()

    diff_mcc = trad_mcc - dl_mcc
    t_mcc, p_mcc = ttest_rel(trad_mcc, dl_mcc)

    # ---------------- FINAL ROW ----------------
    ttest_rows.append({
        "Country": country,
        "Model Comparison": f"{ml_model} vs {dl_model}",

        "Δ Acc": diff_acc.mean(),
        "SD Acc": diff_acc.std(ddof=1),
        "t Acc": t_acc,
        "p Acc": p_acc,

        "Δ MCC": diff_mcc.mean(),
        "SD MCC": diff_mcc.std(ddof=1),
        "t MCC": t_mcc,
        "p MCC": p_mcc,
    })
# ---------- Summary statistics ----------
print("\n================ Summary statistics ================\n")
df_ttest = pd.DataFrame(ttest_rows)
df_ttest


================ Summary statistics ================



,Country,Model Comparison,Δ Acc,SD Acc,t Acc,p Acc,Δ MCC,SD MCC,t MCC,p MCC
0,MYANMAR,CAT vs NN_TORCH,0.00306,0.004456,2.171527,0.057969,0.005127,0.012611,1.285544,0.230694


In [12]:
save_confirm = True   # change to False if don't want to save yet
if save_confirm:
    country_folder = rf"{RESULTS_FOLDER}\{country}"
    os.makedirs(country_folder, exist_ok=True)
    
    df_ttest.to_csv(rf"{country_folder}\Table9_myanmar_best_model_t_test_wealth_inclusive.csv", index=False)
    print("\nFiles saved successfully.")


Files saved successfully.
